# 01 — Scrape (full run)

Collects a sample of public GitHub repos that have a root-level `CLAUDE.md` or, failing that, an `AGENTS.md`, plus repo metadata. `N_TARGET` candidates are scraped (default 1250, oversampled so ~1,000 survive exclusions and `02_clean`). Set the `N_TARGET` environment variable for a smaller smoke test.

Scraping decisions (copy into the dataset card):
- **Unit = repo. `CLAUDE.md` takes priority; `AGENTS.md` is used only when a repo has no `CLAUDE.md`** (or its `CLAUDE.md` is a pointer, resolved in `02_clean`). `primary_file` records the group.
- **Population and sampling.** Code search `filename:<file> path:/ size:<lo>..<hi>` over 23 log-spaced file-size bins (0 B to 1 MB). Each bin's `total_count` gives the size distribution of the searchable population. Candidates are allocated to bins **in proportion to those counts**, so file-length statistics are not distorted by the bucketing used to get past the 1,000-result cap. The two file types are allocated in proportion to their total counts, so the `AGENTS.md`-only share is natural, not a quota.
- Within a bin, random pages of the (at most 1,000) visible results are sampled; results beyond the first 1,000 per bin are invisible to the API. GitHub's `total_count` is approximate.
- **No star/date stratification:** code search has no star or date qualifier, so stars and dates are recorded as covariates, not sampling strata. (Deviation from the original brief, which assumed star/date buckets were possible.)
- Exclusions: forks, archived repos, files over 1 MB, repos with neither file at the root, repos that error out. Every exclusion reason is logged.
- `has_tests` = a test-like directory at the repo root (`test`, `tests`, `__tests__`, `spec`, `specs`). `has_ci` = `.github/workflows`, `.travis.yml`, `.circleci`, `.gitlab-ci.yml`, `azure-pipelines.yml` or `Jenkinsfile` at the root.
- Both files are fetched when a repo has both (`claude_text`, `agents_text`), plus `*_symlink_target` for symlinks. `content_raw` is the primary file's text. Redirect detection happens in `02_clean`.
- Rate limits: code search sleeps 7 s between calls; REST calls wait for the hourly reset when it is hit. Search pages and every scraped repo are cached under `data/raw/cache/`, so an interrupted run resumes where it stopped.

In [1]:
import os, time, json, math, random, base64, datetime as dt
from pathlib import Path
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
TOKEN = os.environ.get("GITHUB_TOKEN")
assert TOKEN, "GITHUB_TOKEN missing from .env"

N_TARGET = int(os.environ.get("N_TARGET", 1250))      # candidate repos to scrape
SEED = 2026
FILE_TYPES = ["CLAUDE.md", "AGENTS.md"]
BOUNDS = [0, 100, 200, 300, 400, 500, 700, 1000, 1400, 2000, 2800, 4000, 5600, 8000, 11000,
          16000, 22000, 32000, 45000, 64000, 90000, 128000, 256000, 1048576]     # bytes
BINS = list(zip(BOUNDS[:-1], BOUNDS[1:]))                                          # [lo, hi)
SEARCH_SLEEP = 7                                       # code search: 10 req/min
STAR_BINS = [(0, 10), (10, 100), (100, 1000), (1000, 10**9)]
RAW = Path("data/raw"); CACHE = RAW / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
OUT_PATH = RAW / f"full_{N_TARGET}.parquet"
SCRAPE_DATE = dt.date.today().isoformat()

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})
print("scrape date:", SCRAPE_DATE, "| N_TARGET:", N_TARGET, "| bins:", len(BINS))

scrape date: 2026-11-10 | N_TARGET: 1250 | bins: 23


In [2]:
def gh(url, params=None, max_errors=6):
    """GET with rate-limit waiting. Returns the Response for 200/404/403(non-rate)/451/410; raises otherwise."""
    errors = 0
    while True:
        try:
            r = session.get(url, params=params, timeout=30)
        except requests.RequestException:
            errors += 1
            if errors > max_errors:
                raise
            time.sleep(2 ** errors)
            continue
        if r.status_code in (403, 429):
            if "retry-after" in r.headers:
                wait = int(r.headers["retry-after"]) + 1
            elif r.headers.get("x-ratelimit-remaining") == "0":
                wait = max(int(r.headers.get("x-ratelimit-reset", time.time() + 60)) - time.time(), 0) + 2
            elif "rate limit" in r.text.lower() or "abuse" in r.text.lower():
                wait = 60
            else:
                return r                                   # e.g. file too large for the contents API
            wait = min(wait, 3700)
            print(f"rate limited, sleeping {wait:.0f}s", flush=True)
            time.sleep(wait)
            continue
        if r.status_code >= 500 or r.status_code == 408:
            errors += 1
            if errors > max_errors:
                raise RuntimeError(f"server errors on {url}")
            time.sleep(2 ** errors)
            continue
        if r.status_code in (404, 410, 451):
            return r
        r.raise_for_status()
        return r

## 1. Search: per (file type, size bin) counts, cached

In [3]:
SEARCH_CACHE = CACHE / "search.jsonl"
search_cache = {}
if SEARCH_CACHE.exists():
    for line in SEARCH_CACHE.open():
        d = json.loads(line); search_cache[d["key"]] = d

def search_page(ft, lo, hi, page):
    key = f"{ft}|{lo}|{hi}|{page}"
    if key in search_cache:
        return search_cache[key]
    r = gh("https://api.github.com/search/code",
           {"q": f"filename:{ft} path:/ size:{lo}..{hi - 1}", "per_page": 100, "page": page})
    if r.status_code != 200:
        raise RuntimeError(f"search failed {r.status_code}: {r.text[:200]}")
    d = r.json()
    rec = {"key": key, "total_count": d.get("total_count", 0),
           "repos": [it["repository"]["full_name"] for it in d.get("items", []) if it["path"] == ft]}
    with SEARCH_CACHE.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    search_cache[key] = rec
    time.sleep(SEARCH_SLEEP)
    return rec

counts = {(ft, lo, hi): search_page(ft, lo, hi, 1)["total_count"] for ft in FILE_TYPES for lo, hi in BINS}
tot = {ft: sum(v for (f, _, _), v in counts.items() if f == ft) for ft in FILE_TYPES}
print("approximate population (GitHub's count, a snapshot):", tot)
pd.DataFrame({ft: {f"{lo}-{hi}": counts[(ft, lo, hi)] for lo, hi in BINS} for ft in FILE_TYPES})

approximate population (GitHub's count, a snapshot): {'CLAUDE.md': 642584, 'AGENTS.md': 642955}


,CLAUDE.md,AGENTS.md
0-100,8624,3752
100-200,10784,5760
200-300,10672,6648
300-400,6992,7392
400-500,7296,7984
500-700,11824,17056
700-1000,15872,29504
1000-1400,24960,41600
1400-2000,35456,51712
2000-2800,60288,65024


## 2. Allocate candidates to bins in proportion to the population, then sample

In [4]:
rng = random.Random(SEED)
T_type = {ft: round(N_TARGET * tot[ft] / sum(tot.values())) for ft in FILE_TYPES}
print("candidates per file type:", T_type)
cands = []
for ft in FILE_TYPES:
    for lo, hi in BINS:
        c = counts[(ft, lo, hi)]
        need = round(T_type[ft] * c / tot[ft]) if c else 0
        if need == 0:
            continue
        max_page = min(10, math.ceil(c / 100))               # the API shows at most 1,000 results
        n_pages = min(max_page, math.ceil(need / 50))
        pool = []
        for p in rng.sample(range(1, max_page + 1), n_pages):
            pool += search_page(ft, lo, hi, p)["repos"]
        pool = list(dict.fromkeys(pool))
        for name in rng.sample(pool, min(need, len(pool))):
            cands.append({"full_name": name, "found_via": ft, "bin_lo": lo, "bin_hi": hi,
                          "bin_count": c, "bin_need": need})

cand = (pd.DataFrame(cands).groupby("full_name", as_index=False)
        .agg(found_via=("found_via", lambda s: "+".join(sorted(set(s)))), bin_lo=("bin_lo", "first"),
             bin_hi=("bin_hi", "first"), bin_count=("bin_count", "first"), bin_need=("bin_need", "first")))
cand.to_parquet(CACHE / f"candidates_{N_TARGET}.parquet", index=False)
print(len(cands), "picks ->", len(cand), "unique repos")

candidates per file type: {'CLAUDE.md': 625, 'AGENTS.md': 625}


1246 picks -> 1246 unique repos


## 3. Per-repo scrape (resumable): metadata, root listing, files, repo signals

In [5]:
TEST_DIRS = {"test", "tests", "__tests__", "spec", "specs"}
CI_ROOT = {".travis.yml", ".circleci", ".gitlab-ci.yml", "azure-pipelines.yml", "jenkinsfile"}
REPO_CACHE = CACHE / "repos.jsonl"
repo_cache = {}
if REPO_CACHE.exists():
    for line in REPO_CACHE.open():
        d = json.loads(line); repo_cache[d["full_name"]] = d

def fetch_file(name, path):
    """Return (text, symlink_target). text is None if missing, a symlink, or >1 MB."""
    f = gh(f"https://api.github.com/repos/{name}/contents/{path}")
    if f.status_code != 200:
        return None, None
    j = f.json()
    if not isinstance(j, dict):
        return None, None
    if j.get("type") == "symlink":
        return None, j.get("target")
    if j.get("encoding") == "base64":
        return base64.b64decode(j["content"]).decode("utf-8", errors="replace"), None
    return None, None

def scrape_repo(name):
    r = gh(f"https://api.github.com/repos/{name}")
    if r.status_code != 200:
        return {"full_name": name, "excluded": f"repo_http_{r.status_code}"}
    m = r.json()
    rec = {"full_name": name, "stars": m["stargazers_count"], "forks": m["forks_count"],
           "language": m["language"], "created_at": m["created_at"], "last_push_at": m["pushed_at"],
           "repo_size_kb": m["size"], "license": (m.get("license") or {}).get("spdx_id"), "excluded": None}
    if m["fork"]:
        rec["excluded"] = "fork"; return rec
    if m["archived"]:
        rec["excluded"] = "archived"; return rec
    root = gh(f"https://api.github.com/repos/{name}/contents/")
    if root.status_code != 200 or not isinstance(root.json(), list):
        rec["excluded"] = "root_unavailable"; return rec
    entries = {e["name"]: e["type"] for e in root.json()}
    lower = {n.lower(): t for n, t in entries.items()}
    rec["has_claude_md"] = "CLAUDE.md" in entries
    rec["has_agents_md"] = "AGENTS.md" in entries
    if not (rec["has_claude_md"] or rec["has_agents_md"]):
        rec["excluded"] = "no_file_at_root"; return rec
    rec["claude_text"], rec["claude_symlink_target"] = fetch_file(name, "CLAUDE.md") if rec["has_claude_md"] else (None, None)
    rec["agents_text"], rec["agents_symlink_target"] = fetch_file(name, "AGENTS.md") if rec["has_agents_md"] else (None, None)
    rec["has_tests"] = any(n in TEST_DIRS and t == "dir" for n, t in lower.items())
    ci = any(n in CI_ROOT for n in lower)
    if ".github" in lower and not ci:
        ci = gh(f"https://api.github.com/repos/{name}/contents/.github/workflows").status_code == 200
    rec["has_ci"] = ci
    return rec

todo = [n for n in cand["full_name"] if n not in repo_cache]
print(len(repo_cache), "cached,", len(todo), "to scrape")
t0, errors = time.time(), 0
for i, name in enumerate(todo, 1):
    try:
        rec = scrape_repo(name)
    except Exception as e:
        rec = {"full_name": name, "excluded": f"error:{type(e).__name__}"}
        errors += 1
        if errors > max(10, 0.05 * len(todo)):
            raise
    repo_cache[name] = rec
    with REPO_CACHE.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    if i % 25 == 0 or i == len(todo):
        msg = f"{i}/{len(todo)} repos, {errors} errors, {(time.time() - t0) / 60:.1f} min"
        (CACHE / "progress.txt").write_text(msg + "\n"); print(msg, flush=True)

37 cached, 1246 to scrape


25/1246 repos, 0 errors, 0.4 min


50/1246 repos, 0 errors, 0.9 min


75/1246 repos, 0 errors, 1.3 min


100/1246 repos, 0 errors, 1.8 min


125/1246 repos, 0 errors, 2.2 min


... (41 outputs left out)

1175/1246 repos, 0 errors, 21.4 min


1200/1246 repos, 0 errors, 21.9 min


1225/1246 repos, 0 errors, 22.3 min


1246/1246 repos, 0 errors, 22.7 min


## 4. Assemble, apply exclusions, checkpoint to Parquet

In [6]:
allrecs = pd.DataFrame([repo_cache[n] for n in cand["full_name"] if n in repo_cache]).merge(cand, on="full_name", how="left")
excl = allrecs[allrecs.excluded.notna()]
sample = allrecs[allrecs.excluded.isna()].copy()
for col in ["has_claude_md", "has_agents_md", "has_tests", "has_ci"]:
    sample[col] = sample[col].astype(bool)
sample["stars"] = sample["stars"].astype(int)
sample["has_both"] = sample["has_claude_md"] & sample["has_agents_md"]
sample["primary_file"] = sample["has_claude_md"].map({True: "CLAUDE.md", False: "AGENTS.md"})

def star_bin(s):
    for lo, hi in STAR_BINS:
        if lo <= s < hi:
            return f"{lo}-{hi if hi < 10**9 else 'inf'}"
sample["star_bin"] = sample["stars"].map(star_bin)

is_claude = sample["primary_file"] == "CLAUDE.md"
sample["content_raw"] = sample["claude_text"].where(is_claude, sample["agents_text"])   # the primary file's text
primary_link = sample["claude_symlink_target"].where(is_claude, sample["agents_symlink_target"])
sample["n_chars"] = sample["content_raw"].str.len()
sample["scraped_at"] = SCRAPE_DATE
keep = sample["content_raw"].notna() | primary_link.notna()          # symlinks have no text of their own
too_big = (~keep).sum()
sample = sample[keep].reset_index(drop=True)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sample.to_parquet(OUT_PATH, index=False)
excl[["full_name", "excluded", "found_via"]].to_parquet(RAW / f"excluded_{N_TARGET}.parquet", index=False)
md_dir = OUT_PATH.with_name(OUT_PATH.stem + "_md"); md_dir.mkdir(exist_ok=True)   # readable copies
for _, row in sample.iterrows():
    if row["content_raw"] is not None:
        (md_dir / f"{row.full_name.replace('/', '__')}__{row.primary_file}").write_text(row["content_raw"])
print("saved", OUT_PATH, sample.shape)

saved data/raw/full_1250.parquet (1246, 28)


## 5. What came out

In [7]:
print(f"candidates {len(cand)} -> scraped {len(allrecs)} -> kept {len(sample)} (files >1 MB / unreadable: {too_big})")
print("\nexclusions:"); print(excl.excluded.value_counts().to_string() if len(excl) else "none")
print("\ngroup (primary_file):"); print(sample.primary_file.value_counts().to_string())
print(f"both files present: {sample.has_both.mean():.0%} | symlinks: {(sample.claude_symlink_target.notna() | sample.agents_symlink_target.notna()).sum()}")
print("\nstar bins:"); print(sample.star_bin.value_counts().sort_index().to_string())
print("\ntop languages:"); print(sample.language.value_counts().head(8).to_string())
print("\nsize-bin check (share of population vs share of sample):")
pop = pd.Series({(lo, hi): sum(counts[(ft, lo, hi)] for ft in FILE_TYPES) for lo, hi in BINS}); pop = pop / pop.sum()
got = sample.groupby(["bin_lo", "bin_hi"]).size(); got = got / got.sum()
pd.DataFrame({"population": pop, "sample": got}).fillna(0).round(3)

candidates 1246 -> scraped 1246 -> kept 1246 (files >1 MB / unreadable: 0)

exclusions:
none

group (primary_file):
primary_file
CLAUDE.md    836
AGENTS.md    410
both files present: 29% | symlinks: 0

star bins:
star_bin
0-10        1112
10-100       105
100-1000      22
1000-inf       7

top languages:
language
TypeScript    301
Python        287
JavaScript    108
Rust           70
Go             56
HTML           51
Shell          29
C#             28

size-bin check (share of population vs share of sample):


,,population,sample
0,100,0.010,0.010
100,200,0.013,0.013
200,300,0.013,0.013
300,400,0.011,0.011
400,500,0.012,0.012
500,700,0.022,0.023
700,1000,0.035,0.035
1000,1400,0.052,0.051
1400,2000,0.068,0.067
2000,2800,0.097,0.098
